# cAPTure: model-ready graph materialization

This notebook converts the prepared five-second development windows into immutable, fold-specific graph shards. It does not train a model and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

Run **SMOKE → manual review → FULL_DEV**. Each packet becomes one directed edge. Parallel edges and self-loops are preserved, empty windows remain represented by gaps in `window_index`, and `decision_time` is the window end.

## 1. Mount Drive and prepare the environment

Push the current branch before opening Colab. This notebook installs the existing CPU data dependencies only; it does not install the training stack or run tests.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_graph_materialization_work")

REPOSITORY_ALREADY_PRESENT = PROJECT_ROOT.exists()
if not REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
if REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )

required_files = [
    "code/python/requirements-capture.txt",
    "code/python/utils/capture_graph_materialization.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_packet_schema_v1.yaml",
    "configs/capture_preprocessing_v1.yaml",
    "configs/capture_graph_materialization_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import numpy as np
import pandas as pd
from IPython.display import display
from utils.capture_graph_materialization import run_capture_graph_materialization

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU graph-materialization environment is ready.")

Mounted at /content/drive
Repository commit: 63c5040cc33771eb1c6e55c8e916cb03ad598d00
CPU graph-materialization environment is ready.


## 2. Bind the approved inputs

These IDs bind materialization to the verified FULL_DEV preparation, the fold-local preprocessing audit, and the Stage-1 decision. If your Drive IDs differ, edit only the three ID constants. A run requires `stage1_decision.md` to contain `PASS_WITH_LIMITATIONS`.

Materializing both feature views may require several GiB on Drive. `/content` is used only for one fold/scenario at a time. Reduce `BATCH_SIZE` if RAM is tight; this does not change the scientific contract.

In [2]:
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_RUN_ID = "20260919T004143_161396Z_preprocessing"
STRUCTURAL_RUN_ID = "20260928T145715_010326Z_graph_structural_full_dev"

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
MATERIALIZATION_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_materialization_v1.yaml"
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
PREPROCESSING_AUDIT_DIR = (
    DRIVE_ROOT / "preprocessing_runs" / PREPROCESSING_RUN_ID
)
STAGE1_DECISION_PATH = (
    DRIVE_ROOT / "graph_pilot_runs" / STRUCTURAL_RUN_ID / "stage1_decision.md"
)
BATCH_SIZE = 100_000

required_inputs = [
    MANIFEST_PATH, PACKET_SCHEMA_PATH, PREPROCESSING_SCHEMA_PATH,
    MATERIALIZATION_CONFIG_PATH, PREPARED_RUN_DIR,
    PREPROCESSING_AUDIT_DIR, STAGE1_DECISION_PATH,
]
for required_path in required_inputs:
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")
stage1_decision_lines = STAGE1_DECISION_PATH.read_text().splitlines()
if "Status: **PASS_WITH_LIMITATIONS**." not in stage1_decision_lines:
    raise RuntimeError("Stage 1 is not recorded as PASS_WITH_LIMITATIONS.")

print("Prepared input:", PREPARED_RUN_DIR)
print("Preprocessing audit:", PREPROCESSING_AUDIT_DIR)
print("Stage-1 decision:", STAGE1_DECISION_PATH)
print("Local staging:", LOCAL_WORK_ROOT)

Prepared input: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Preprocessing audit: /content/drive/MyDrive/capture_gate0/preprocessing_runs/20260919T004143_161396Z_preprocessing
Stage-1 decision: /content/drive/MyDrive/capture_gate0/graph_pilot_runs/20260928T145715_010326Z_graph_structural_full_dev/stage1_decision.md
Local staging: /content/capture_graph_materialization_work


## 3. Run SMOKE

SMOKE materializes `train_empty_conn` and `train_dollar_char` under both fold preprocessors. Leave the ID as `None` for a new run. To resume after an interruption or Colab restart, paste the previously printed ID.

In [3]:
SMOKE_RUN_ID = "20260928T170548_975946Z_graph_materialization_smoke"
if SMOKE_RUN_ID is None:
    SMOKE_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_materialization_smoke"
    )
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "graph_materialization_runs" / SMOKE_RUN_ID
print("SMOKE run ID:", SMOKE_RUN_ID)
print("SMOKE output:", SMOKE_OUTPUT_DIR)

SMOKE_RESULT = run_capture_graph_materialization(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    materialization_config_path=MATERIALIZATION_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
    stage1_decision_path=STAGE1_DECISION_PATH,
    output_dir=SMOKE_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="SMOKE",
    batch_size=BATCH_SIZE,
)

SMOKE run ID: 20260928T170548_975946Z_graph_materialization_smoke
SMOKE output: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T170548_975946Z_graph_materialization_smoke
Validating completed fold A/train_empty_conn...
Validating completed fold A/train_dollar_char...
Validating completed fold B/train_empty_conn...
Validating completed fold B/train_dollar_char...
Graph materialization complete: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T170548_975946Z_graph_materialization_smoke


## 4. Review SMOKE

The cross-fold checks must pass. `feature_tensors_differ=True` is expected because each fold has a separately fitted preprocessor; topology, labels, source rows, feature names, and semantic node mapping must remain identical. This review also scans the stored SMOKE features and compares their nonzero counts and maximum absolute values with the completed preprocessing audit. Inspect shard counts, compression, largest graph, runtime, and peak RAM before continuing.

In [4]:
def materialization_table(run_dir):
    manifest = json.loads(
        (run_dir / "graph_materialization_manifest.json").read_text()
    )
    rows = []
    for fold in ("A", "B"):
        for scenario in manifest["scenarios"]:
            report = json.loads(
                (run_dir / f"fold_{fold}" / scenario
                 / "scenario_fold_report.json").read_text()
            )
            rows.append({
                "fold": fold,
                "scenario": scenario,
                "graphs": report["graphs"],
                "edges": report["edges"],
                "nodes": report["distinct_global_nodes"],
                "feature_dim": report["feature_dim"],
                "shards": report["shard_count"],
                "compressed_GiB": report["compressed_shard_bytes"] / 1024**3,
                "wall_seconds": report["wall_seconds"],
                "peak_RSS_GiB": (
                    report["peak_sampled_rss_bytes"] / 1024**3
                    if report["peak_sampled_rss_bytes"] is not None else None
                ),
            })
    return manifest, pd.DataFrame(rows)

smoke_manifest, smoke_table = materialization_table(SMOKE_OUTPUT_DIR)
display(smoke_table)
display(pd.DataFrame.from_dict(
    smoke_manifest["cross_fold_invariants"], orient="index"
))
print(json.dumps(smoke_manifest["totals"], indent=2))
print("Held-out scenarios accessed:",
      smoke_manifest["provenance"]["held_out_scenarios_accessed"])
print("Training performed:",
      smoke_manifest["provenance"]["training_performed"])

preprocessing_audit = json.loads(
    (PREPROCESSING_AUDIT_DIR / "capture_preprocessing_audit.json").read_text()
)
feature_integrity_rows = []
for fold in ("A", "B"):
    for scenario in smoke_manifest["scenarios"]:
        scenario_dir = SMOKE_OUTPUT_DIR / f"fold_{fold}" / scenario
        report = json.loads(
            (scenario_dir / "scenario_fold_report.json").read_text()
        )
        expected = preprocessing_audit["folds"][fold]["scenario_audits"][scenario]
        observed_nonzero = 0
        all_zero_edges = 0
        maximum_absolute_value = 0.0
        maximum_edges_in_graph = 0
        finite_values = True
        for shard_record in report["shards"]:
            with np.load(scenario_dir / shard_record["path"], allow_pickle=False) as shard:
                edge_attr = shard["edge_attr"]
                observed_nonzero += int(np.count_nonzero(edge_attr))
                all_zero_edges += int(np.count_nonzero(np.all(edge_attr == 0, axis=1)))
                if edge_attr.size:
                    maximum_absolute_value = max(
                        maximum_absolute_value, float(np.abs(edge_attr).max())
                    )
                finite_values = finite_values and bool(np.isfinite(edge_attr).all())
                maximum_edges_in_graph = max(
                    maximum_edges_in_graph, int(np.diff(shard["edge_ptr"]).max())
                )
        total_values = report["edges"] * report["feature_dim"]
        feature_integrity_rows.append({
            "fold": fold,
            "scenario": scenario,
            "active_features": preprocessing_audit["folds"][fold]["active_feature_count"],
            "nonzero_value_fraction": observed_nonzero / total_values,
            "all_zero_edge_fraction": all_zero_edges / report["edges"],
            "maximum_absolute_value": maximum_absolute_value,
            "maximum_edges_in_graph": maximum_edges_in_graph,
            "edge_attr_to_compressed_ratio": (
                total_values * np.dtype(np.float32).itemsize
                / report["compressed_shard_bytes"]
            ),
            "finite_values": finite_values,
            "nonzero_count_matches_preprocessing": (
                observed_nonzero == expected["nonzero_values"]
            ),
            "maximum_matches_preprocessing": (
                maximum_absolute_value == expected["maximum_absolute_value"]
            ),
            "oversized_single_graph_shards": report["oversized_single_graph_shards"],
        })

feature_integrity = pd.DataFrame(feature_integrity_rows)
display(feature_integrity)
required_feature_checks = [
    "finite_values",
    "nonzero_count_matches_preprocessing",
    "maximum_matches_preprocessing",
]
if not feature_integrity[required_feature_checks].all(axis=None):
    raise RuntimeError("Stored edge features differ from the preprocessing audit.")

,fold,scenario,graphs,edges,nodes,feature_dim,shards,compressed_GiB,wall_seconds,peak_RSS_GiB
0,A,train_empty_conn,9360,1175779,160,103,37,0.009863,43.600000,0.509247
1,A,train_dollar_char,9116,2882555,160,103,36,0.023473,75.223702,0.524948
2,B,train_empty_conn,9360,1175779,160,103,37,0.009854,43.330908,0.587574
3,B,train_dollar_char,9116,2882555,160,103,36,0.023448,76.619996,0.534771


,topology_and_labels_identical,feature_tensors_differ,graphs,edges,nodes
train_empty_conn,True,True,9360,1175779,160
train_dollar_char,True,True,9116,2882555,160


{
  "unique_development_packets": 4058334,
  "materialized_edges_across_folds": 8116668,
  "graphs_across_folds": 36952,
  "compressed_shard_bytes": 71550982,
  "shards": 146
}
Held-out scenarios accessed: False
Training performed: False


,fold,scenario,active_features,nonzero_value_fraction,all_zero_edge_fraction,maximum_absolute_value,maximum_edges_in_graph,edge_attr_to_compressed_ratio,finite_values,nonzero_count_matches_preprocessing,maximum_matches_preprocessing,oversized_single_graph_shards
0,A,train_empty_conn,89,0.130784,0.0,4.349609,42145,45.743805,True,True,True,0
1,A,train_dollar_char,89,0.137619,0.0,4.349609,43177,47.120564,True,True,True,0
2,B,train_empty_conn,92,0.130784,0.0,4.771740,42145,45.783389,True,True,True,0
3,B,train_dollar_char,92,0.137619,0.0,4.771740,43177,47.171109,True,True,True,0


## 5. Manual approval for FULL_DEV

Set `APPROVE_FULL_DEV=True` only after the SMOKE manifest and every row above are plausible. Record a concrete note; it becomes part of the notebook evidence, while the immutable Stage-1 decision remains the scientific gate.

In [5]:
APPROVE_FULL_DEV = True
SMOKE_REVIEW_NOTE = (
    "Packet-edge conservation passed exactly. Cross-fold topology, labels, "
    "source rows, feature order, and node mappings were identical, while "
    "fold-specific feature tensors differed as expected. Stored nonzero counts "
    "and maximum values matched the preprocessing audit exactly. Every edge "
    "retained at least one nonzero feature, all values were finite, and no "
    "oversized single-graph shards were produced. Runtime, memory, compression, "
    "and maximum graph size were acceptable."
)

if not APPROVE_FULL_DEV:
    raise RuntimeError(
        "Review SMOKE, then set APPROVE_FULL_DEV=True and record the note."
    )
if not SMOKE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SMOKE_REVIEW_NOTE before FULL_DEV.")
print("FULL_DEV materialization approved from SMOKE:", SMOKE_RUN_ID)
print(SMOKE_REVIEW_NOTE)

FULL_DEV materialization approved from SMOKE: 20260928T170548_975946Z_graph_materialization_smoke
Packet-edge conservation passed exactly. Cross-fold topology, labels, source rows, feature order, and node mappings were identical, while fold-specific feature tensors differed as expected. Stored nonzero counts and maximum values matched the preprocessing audit exactly. Every edge retained at least one nonzero feature, all values were finite, and no oversized single-graph shards were produced. Runtime, memory, compression, and maximum graph size were acceptable.


## 6. Run FULL_DEV

FULL_DEV writes both fold-specific feature views for all five development scenarios. Completed fold/scenario directories are checksum-validated when resuming. Keep the printed run ID.

In [6]:
FULL_DEV_RUN_ID = None  # Paste an existing ID here to resume.
if FULL_DEV_RUN_ID is None:
    FULL_DEV_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_materialization_full_dev"
    )
FULL_DEV_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / FULL_DEV_RUN_ID
)
print("FULL_DEV run ID:", FULL_DEV_RUN_ID)
print("FULL_DEV output:", FULL_DEV_OUTPUT_DIR)

FULL_DEV_RESULT = run_capture_graph_materialization(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    materialization_config_path=MATERIALIZATION_CONFIG_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
    stage1_decision_path=STAGE1_DECISION_PATH,
    output_dir=FULL_DEV_OUTPUT_DIR,
    local_work_root=LOCAL_WORK_ROOT,
    mode="FULL_DEV",
    batch_size=BATCH_SIZE,
)

FULL_DEV run ID: 20260928T175821_521784Z_graph_materialization_full_dev
FULL_DEV output: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T175821_521784Z_graph_materialization_full_dev
Staging fold A/train_empty_conn...
fold A/train_empty_conn: 2,500 graphs, 591,454 edges, 9 completed shards
fold A/train_empty_conn: 5,000 graphs, 815,365 edges, 19 completed shards
fold A/train_empty_conn: 7,500 graphs, 1,037,003 edges, 29 completed shards
Completed fold A/train_empty_conn: 9,360 graphs, 1,175,779 edges, 37 shards
Staging fold A/train_qos_mid...
fold A/train_qos_mid: 2,500 graphs, 283,124 edges, 9 completed shards
fold A/train_qos_mid: 5,000 graphs, 825,890 edges, 19 completed shards
fold A/train_qos_mid: 7,500 graphs, 1,360,941 edges, 29 completed shards
Completed fold A/train_qos_mid: 9,360 graphs, 1,499,717 edges, 37 shards
Staging fold A/train_dollar_char...
fold A/train_dollar_char: 2,500 graphs, 912,791 edges, 9 completed shards
fold A/train_dollar_char: 5,0

## 7. Review FULL_DEV

Do not start training until every cross-fold invariant is present and true, the unique packet total matches preparation, and resource use is acceptable. The next implementation step will be the shard loader plus the frozen one-seed training contract; it will consume this run ID without rebuilding graphs.

In [7]:
full_manifest, full_table = materialization_table(FULL_DEV_OUTPUT_DIR)
full_invariants = pd.DataFrame.from_dict(
    full_manifest["cross_fold_invariants"], orient="index"
)
display(full_table)
display(full_invariants)
print(json.dumps(full_manifest["totals"], indent=2))
print("Compressed shards (GiB):",
      full_manifest["totals"]["compressed_shard_bytes"] / 1024**3)
print("Manifest:",
      FULL_DEV_OUTPUT_DIR / "graph_materialization_manifest.json")

if not full_invariants["topology_and_labels_identical"].all():
    raise RuntimeError("Cross-fold topology or label invariants failed.")
if full_manifest["provenance"]["held_out_scenarios_accessed"]:
    raise RuntimeError("A held-out scenario was accessed.")
if full_manifest["provenance"]["training_performed"]:
    raise RuntimeError("Training unexpectedly occurred during materialization.")

,fold,scenario,graphs,edges,nodes,feature_dim,shards,compressed_GiB,wall_seconds,peak_RSS_GiB
0,A,train_empty_conn,9360,1175779,160,103,37,0.009863,56.025145,0.576187
1,A,train_qos_mid,9360,1499717,163,103,37,0.012706,70.232457,0.473976
2,A,train_dollar_char,9116,2882555,160,103,36,0.023473,92.297215,0.533176
3,A,train_slash_char,9116,6425516,164,103,36,0.047577,196.158385,0.623722
4,A,train_sub_exf,9115,2225807,121,103,36,0.019356,85.381610,0.622139
5,B,train_empty_conn,9360,1175779,160,103,37,0.009854,57.010709,0.637226
6,B,train_qos_mid,9360,1499717,163,103,37,0.012684,63.642015,0.566254
7,B,train_dollar_char,9116,2882555,160,103,36,0.023448,101.190029,0.597122
8,B,train_slash_char,9116,6425516,164,103,36,0.047465,164.059870,0.630596
9,B,train_sub_exf,9115,2225807,121,103,36,0.019342,92.552575,0.613411


,topology_and_labels_identical,feature_tensors_differ,graphs,edges,nodes
train_empty_conn,True,True,9360,1175779,160
train_qos_mid,True,True,9360,1499717,163
train_dollar_char,True,True,9116,2882555,160
train_slash_char,True,True,9116,6425516,164
train_sub_exf,True,True,9115,2225807,121


{
  "unique_development_packets": 14209374,
  "materialized_edges_across_folds": 28418748,
  "graphs_across_folds": 92134,
  "compressed_shard_bytes": 242415458,
  "shards": 364
}
Compressed shards (GiB): 0.2257669884711504
Manifest: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T175821_521784Z_graph_materialization_full_dev/graph_materialization_manifest.json
